In [3]:
import os
import json
import random
import re
import boto3
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# MongoDB connection details
MONGODB_URI = os.getenv('MONGODB_URI_REALM')
DB_NAME = 'gpt'
COLLECTION_NAME = 'list'

# Cloudflare R2 bucket details
R2_ACCESS_KEY = os.getenv('CLOUDFLARE_ACCESS_KEY')
R2_SECRET_KEY = os.getenv('CLOUDFLARE_SECRET_KEY')
R2_BUCKET = os.getenv('CLOUDFLARE_BUCKET')
R2_ENDPOINT = os.getenv('CLOUDFLARE_ENDPOINT')
print(R2_ACCESS_KEY, R2_SECRET_KEY, R2_BUCKET, R2_ENDPOINT)

# Slugify function to convert strings to URL-friendly slugs
def slugify(value):
    if not value:
        return None
    value = value.lower()
    value = re.sub(r'[^\w\s-]', '', value)  # Remove non-word characters
    value = re.sub(r'[\s_]+', '-', value)  # Replace spaces and underscores with hyphens
    return value

# Upload files to Cloudflare R2 bucket
def upload_to_r2(local_file_path, r2_key):
    session = boto3.session.Session()
    s3 = session.client(
        's3',
        endpoint_url=R2_ENDPOINT,
        aws_access_key_id=R2_ACCESS_KEY,
        aws_secret_access_key=R2_SECRET_KEY,
    )
    print (f"Uploading {local_file_path} to R2 as {r2_key}")
    # Check if the file exists and is not empty
    if not os.path.exists(local_file_path):
        print(f"Error: {local_file_path} does not exist.")
        return
    if os.path.getsize(local_file_path) == 0:
        print(f"Error: {local_file_path} is empty.")
        return

    try:
        s3.upload_file(local_file_path, R2_BUCKET, r2_key)
        print(f"Successfully uploaded {local_file_path} to R2 as {r2_key}")
    except Exception as e:
        print(f"Error uploading {local_file_path}: {str(e)}")

# Connect to MongoDB
from pymongo import MongoClient
client = MongoClient(MONGODB_URI)
db = client[DB_NAME]
collection = db[COLLECTION_NAME]

# Fetch all documents from the collection
documents = list(collection.find({}))

# Group documents by postid
postid_grouped = {}
for doc in documents:
    postid = doc.get("postid")
    if postid not in postid_grouped:
        postid_grouped[postid] = []
    # Use title as a fallback if slug is missing
    slug = slugify(doc.get("slug")) if doc.get("slug") else slugify(doc.get("title"))
    postid_grouped[postid].append({
        "_id": str(doc["_id"]),
        "title": doc.get("title"),
        "slug": slug,
        "excerpt": doc.get("excerpt"),
        "outline": doc.get("outline"),
        "exists": True if doc.get("content") else False
    })

# Create local 'feed' folder
os.makedirs("feed", exist_ok=True)

# Save grouped files by postid and upload to Cloudflare R2
for postid, docs in postid_grouped.items():
    local_file_path = f"feed/{postid}.json"
    with open(local_file_path, "w") as f:
        json.dump(docs, f, indent=4)
    # Upload to R2
    upload_to_r2(local_file_path, f"feed/{postid}.json")


# Group documents by category
category_grouped = {}
for doc in documents:
    category = doc.get("category", "uncategorized").lower()  # Convert category to lowercase
    slugified_category = slugify(category)  # Slugify category
    if slugified_category not in category_grouped:
        category_grouped[slugified_category] = []
    # Use title as a fallback if slug is missing
    slug = slugify(doc.get("slug")) if doc.get("slug") else slugify(doc.get("title"))
    category_grouped[slugified_category].append({
        "_id": str(doc["_id"]),
        "title": doc.get("title"),
        "slug": slug,
        "excerpt": doc.get("excerpt"),
        "outline": doc.get("outline"),
        "exists": True if doc.get("content") else False
    })
# Shuffle and limit the items to 25, then upload each category file to Cloudflare R2
for category, docs in category_grouped.items():
    random.shuffle(docs)
    limited_docs = docs[:25]
    local_file_path = f"feed/{category}.json"
    with open(local_file_path, "w") as f:
        json.dump(limited_docs, f, indent=4)
    # Upload to R2
    upload_to_r2(local_file_path, f"feed/{category}.json")

print("Files have been created and uploaded to Cloudflare R2 successfully.")

40b2024f74db2640dc1f6ece158c44db 0e4681b50199c289dadb6f54e4177b9c53c80ecbb51e26fc4e41af125227f94e api https://cbe44b125e21f7aada0eefba2df8fc30.r2.cloudflarestorage.com
Uploading feed/None.json to R2 as api/feed/None.json
feed/None.json
api/feed/None.json
api
Successfully uploaded feed/None.json to R2 as api/feed/None.json
Uploading feed/54f146bf29ee86141a496042.json to R2 as api/feed/54f146bf29ee86141a496042.json
feed/54f146bf29ee86141a496042.json
api/feed/54f146bf29ee86141a496042.json
api
Successfully uploaded feed/54f146bf29ee86141a496042.json to R2 as api/feed/54f146bf29ee86141a496042.json
Uploading feed/5985e4f2ef250a1b038b4567.json to R2 as api/feed/5985e4f2ef250a1b038b4567.json
feed/5985e4f2ef250a1b038b4567.json
api/feed/5985e4f2ef250a1b038b4567.json
api
Successfully uploaded feed/5985e4f2ef250a1b038b4567.json to R2 as api/feed/5985e4f2ef250a1b038b4567.json
Uploading feed/53655675aab5a3eb796c927a.json to R2 as api/feed/53655675aab5a3eb796c927a.json
feed/53655675aab5a3eb796c927a.j

: 